# Robustness and interpretation of the modality pilot

Read `small_GRN-modalities.ipynb` first. This notebook checks what the headline percentages
depend on. The same toy networks, sampled pairs and stored simulations are reused throughout.
The primary questions are: does a comparison survive matching denominators; is the signal
concentrated among shared responsive targets; and how much depends on the actuator model?

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
assert (ROOT / 'src' / 'grn.py').exists(), 'Open this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'src'))
from grn import grn
from modality_analysis import (
    CONDITIONS, make_network, choose_pairs, screen, load_screen, residuals,
    summarize, shared_downstream, load_network, save_network, run_conditions
)
threadpool_limits(limits=1)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
N_GENES = 256
RESULTS = ROOT / 'results' / f'modalities_{N_GENES}'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
SEEDS = {'V1': 42, 'V3': 5}
CORE = ['KO', 'KD_90pct', 'KD_50pct', 'CRISPRa_2x', 'CRISPRa_10x', 'OE_10x', 'OE_30x']
def get_data(version, condition):
    return load_screen(RESULTS / f'{version}_seed{SEEDS[version]}' / condition / f'{condition}.npz')
def get_network(version):
    return load_network(RESULTS / f'{version}_seed{SEEDS[version]}' / 'network.npz')

In [2]:
summary = pd.read_csv(RESULTS / 'summary.csv')
networks = {v: get_network(v) for v in SEEDS}
display(pd.read_csv(RESULTS / 'network_diagnostics.csv').round(5))

,version,seed,converged,max_real_eigenvalue,functional_edges,regulators,wt_above_floor
0,V1,42,True,-0.37507,820,120,256
1,V1,5,True,-0.13811,864,118,256
2,V1,7,True,-0.31926,738,114,256
3,V3,0,False,0.05533,13104,253,218
4,V3,1,False,-0.13387,10934,249,216
5,V3,2,False,-0.07232,11517,247,211
6,V3,3,True,-0.02652,10399,250,211
7,V3,4,False,0.24109,11197,249,230
8,V3,5,True,-0.09200,10436,251,210
9,V3,6,False,0.40084,11428,251,217


The toy V3 generator changes both edge density and out-degree heterogeneity relative to V1;
it is not a controlled manipulation of one buffering mechanism. Only converged WT states
have a meaningful equilibrium-stability interpretation. The diagnostic Jacobian is
`(p*(1-p))[:,None] * beta.T - diag(decay)`, where `p=expit(alpha.ravel()+wt@beta)`.
The original V3 notebook broadcasts a one-dimensional `alpha` against a column vector,
which produces an unintended matrix before constructing its Jacobian.

The V3 seed scan covers seeds 0–9 with fixed solver settings. Additional stable-WT seeds
3, 7 and 9 are included below as **stability-conditioned** sensitivity checks, not an unbiased
sample of all V3 networks. V1 seeds 5 and 7 provide additional sparse-network comparisons.

## Same pairs and same measured genes across modalities

In [3]:
matched_rows = []
for version in SEEDS:
    analyses = {c: residuals(get_data(version, c)) for c in CORE}
    common = np.logical_and.reduce([a['mask'] for a in analyses.values()])
    usable_pairs = common.any(axis=1)
    for name, a in analyses.items():
        hits = (np.abs(a['log']) > .1) & common
        matched_rows.append({'network': version, 'condition': name,
            'common_pairs': int(usable_pairs.sum()), 'common_readouts': int(common.sum()),
            'readouts_nonadditive_pct': 100*hits.sum()/common.sum(),
            'pairs_any_nonadditive_pct': 100*hits.any(axis=1)[usable_pairs].mean()})
matched = pd.DataFrame(matched_rows)
matched.to_csv(RESULTS / 'matched_denominators.csv', index=False)
display(matched.round(3))

,network,condition,common_pairs,common_readouts,readouts_nonadditive_pct,pairs_any_nonadditive_pct
0,V1,KO,4096,1032398,0.115,1.733
1,V1,KD_90pct,4096,1032398,0.087,1.416
2,V1,KD_50pct,4096,1032398,0.011,0.366
3,V1,CRISPRa_2x,4096,1032398,0.053,1.123
4,V1,CRISPRa_10x,4096,1032398,0.688,7.739
5,V1,OE_10x,4096,1032398,0.562,5.591
6,V1,OE_30x,4096,1032398,1.076,8.325
7,V3,KO,1038,207852,0.446,2.697
8,V3,KD_90pct,1038,207852,0.392,2.697
9,V3,KD_50pct,1038,207852,0.191,1.445


## Effect-size thresholds, expression floors, and the modified genes

In [4]:
threshold_table = summary[summary.condition.isin(CORE)][['network','condition',
    'readout_gt_0.01','readout_gt_0.1','readout_gt_0.5','readout_gt_1']].copy()
threshold_table.iloc[:, 2:] *= 100
display(threshold_table.round(4))

sensitivity_rows = []
settings = [(1e-4, 'all', True), (1e-6, 'all', True), (0, 'all', True),
            (1e-4, 'wt', True), (1e-4, 'all', False)]
for version in SEEDS:
    for condition in CORE:
        data = get_data(version, condition)
        for floor, policy, exclude in settings:
            out = summarize(data, expression_floor=floor, floor_policy=policy, exclude_targets=exclude)
            out['condition'] = condition
            sensitivity_rows.append(out)
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(RESULTS / 'mask_sensitivity.csv', index=False)
view = sensitivity[['network','condition','expression_floor','floor_policy','exclude_targets',
                    'n_readouts','readout_gt_0.1','pair_any_gt_0.1','linear_readout_gt_0.1']].copy()
display(view[view.condition.isin(['KO','CRISPRa_10x','OE_30x'])].round(5))

,network,condition,readout_gt_0.01,readout_gt_0.1,readout_gt_0.5,readout_gt_1
0,V1,CRISPRa_10x,2.4329,0.7273,0.1427,0.0534
2,V1,CRISPRa_2x,0.5909,0.0704,0.0048,0.0011
5,V1,KD_50pct,0.2854,0.0183,0.0007,0.0001
6,V1,KD_90pct,0.6592,0.1091,0.0118,0.0031
8,V1,KO,0.7529,0.1415,0.0171,0.0042
9,V1,OE_10x,2.1941,0.5911,0.0972,0.0322
11,V1,OE_30x,3.1136,1.0767,0.2256,0.0879
19,V3,CRISPRa_10x,23.8780,14.8418,9.3386,7.1729
21,V3,CRISPRa_2x,11.2997,4.5120,1.8426,1.1652
24,V3,KD_50pct,9.3388,3.3341,1.2178,0.7354


,network,condition,expression_floor,floor_policy,exclude_targets,n_readouts,readout_gt_0.1,pair_any_gt_0.1,linear_readout_gt_0.1
0,V1,KO,0.0001,all,True,1040384,0.00141,0.02002,0.00111
1,V1,KO,0.0000,all,True,1040384,0.00141,0.02002,0.00111
2,V1,KO,0.0000,all,True,1040384,0.00141,0.02002,0.00111
3,V1,KO,0.0001,wt,True,1040384,0.00141,0.02002,0.00111
4,V1,KO,0.0001,all,False,1040384,0.00141,0.02002,0.00111
20,V1,CRISPRa_10x,0.0001,all,True,1037051,0.00727,0.08618,0.00575
21,V1,CRISPRa_10x,0.0000,all,True,1038264,0.00744,0.09082,0.00585
22,V1,CRISPRa_10x,0.0000,all,True,1040384,0.00777,0.09668,0.00612
23,V1,CRISPRa_10x,0.0001,wt,True,1040384,0.00777,0.09668,0.00612
24,V1,CRISPRa_10x,0.0001,all,False,1045233,0.00729,0.08618,0.00688


The “include targets” check keeps finite values only; KO target logFC is undefined at zero
and is still excluded. Therefore it cannot estimate an all-target KO log statistic.
The raw-scale interaction is $(x_{AB}-x_A-x_B+x_{WT})/x_{WT}$. Its 0.1 threshold means
10% of WT expression and has different units from the 0.1-log2 threshold. Comparing the
two detects dependence on the null model, not statistical significance.

## Shared descendants versus shared measurable single effects

In [5]:
overlap = summary[summary.condition.isin(CORE)][['network','condition','n_readouts',
    'shared_downstream_n','shared_downstream_gt_0.1','shared_effective_n','shared_effective_gt_0.1']].copy()
overlap['shared_downstream_gt_0.1'] *= 100
overlap['shared_effective_gt_0.1'] *= 100
display(overlap.round(3))
# Effective overlap: BOTH single perturbations change this readout by >0.1 log2,
# using the same target exclusion, expression floor and convergence mask.

,network,condition,n_readouts,shared_downstream_n,shared_downstream_gt_0.1,shared_effective_n,shared_effective_gt_0.1
0,V1,CRISPRa_10x,1037051,93460,8.070,8179,62.832
2,V1,CRISPRa_2x,1040138,94457,0.775,1506,30.876
5,V1,KD_50pct,1040384,94544,0.201,792,19.444
6,V1,KD_90pct,1040384,94544,1.200,1706,51.465
8,V1,KO,1040384,94544,1.557,1901,58.338
9,V1,OE_10x,1037278,93519,6.556,8628,57.673
11,V1,OE_30x,1032405,92280,12.046,12893,67.510
19,V3,CRISPRa_10x,485050,457994,15.719,47816,94.044
21,V3,CRISPRa_2x,572873,544509,4.747,20824,74.649
24,V3,KD_50pct,631376,600507,3.506,18967,67.528


Sharing a path is not sufficient for a large interaction. Regulatory signals can be small,
cancel, or enter a regime where the measured response is almost linear. For a feed-forward
target with sigmoid input $z$ and two input changes $u,v$, the logFC interaction is

$$h(z+u+v)-h(z+u)-h(z+v)+h(z),\quad h(z)=\log_2\sigma(z).$$

For small changes this is approximately
$-\sigma(z)(1-\sigma(z))uv/\ln 2$. Shared input can therefore give an arbitrarily small
interaction, especially far into a saturation regime or for weak doses. This supports
testing buffering, but the observed rarity alone does not uniquely establish its cause.

In [6]:
from scipy.special import log_expit
z = np.linspace(-12, 12, 500)
h = lambda x: log_expit(x)/np.log(2)
fig, ax = plt.subplots(figsize=(7, 3.6))
for dose in [.1, .5, 1.]:
    interaction = h(z+2*dose)-2*h(z+dose)+h(z)
    ax.plot(z, interaction, label=f'u = v = {dose}')
ax.set(xlabel='WT sigmoid input z', ylabel='Interaction residual (log2)',
       title='Two shared inputs need not produce a large interaction')
ax.axhline(-.1, color='grey', ls=':', lw=1)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / 'shared_input_motif.png', bbox_inches='tight')
plt.show()

<Figure size 840x432 with 1 Axes>

## Actuator assumptions and achieved target expression

In [7]:
names = ['KD_90pct','clamp_0p1x','CRISPRa_2x','promoter_2x','OE_2x','clamp_2x',
         'CRISPRa_10x','promoter_10x','OE_10x','clamp_10x','OE_30x','clamp_30x']
display(summary[summary.condition.isin(names)][['network','condition','dose',
    'achieved_fold_p10','achieved_fold_p50','achieved_fold_p90','target_cap_fraction',
    'readout_gt_0.1','pair_any_gt_0.1']].round(4))

,network,condition,dose,achieved_fold_p10,achieved_fold_p50,achieved_fold_p90,target_cap_fraction,readout_gt_0.1,pair_any_gt_0.1
0,V1,CRISPRa_10x,10.0,9.9955,10.0000,10.0867,0.0000,0.0073,0.0862
2,V1,CRISPRa_2x,2.0,1.9999,2.0000,2.0034,0.0000,0.0007,0.0142
6,V1,KD_90pct,0.1,0.0998,0.1000,0.1000,0.0000,0.0011,0.0154
9,V1,OE_10x,10.0,9.9996,10.0000,10.0087,0.0000,0.0059,0.0574
10,V1,OE_2x,2.0,2.0000,2.0000,2.0017,0.0000,0.0003,0.0076
11,V1,OE_30x,30.0,29.9992,30.0000,30.0168,0.0000,0.0108,0.0833
12,V1,clamp_0p1x,0.1,0.1000,0.1000,0.1000,0.0000,0.0012,0.0168
14,V1,clamp_10x,10.0,10.0000,10.0000,10.0000,0.0000,0.0062,0.0625
15,V1,clamp_2x,2.0,2.0000,2.0000,2.0000,0.0000,0.0009,0.0142
16,V1,clamp_30x,30.0,30.0000,30.0000,30.0000,0.0000,0.0107,0.0850


The original sigmoid limits endogenous synthesis to 1. Raising only `alpha` cannot produce
arbitrarily high expression from a gene already near that ceiling. The promoter variant
explicitly caps unattainable requested folds and reports the capped fraction. The capacity
variant and transgene source can exceed the original ceiling. The clamp variant sets the
achieved RNA dose exactly, eliminating feedback on the perturbed RNA itself.

Two named techniques implemented as the same clamp at the same dose are mathematically
identical in this GRN. Any technique-specific conclusion requires an additional actuator
assumption or measurements of achieved expression. Guide variability, off-targets, isoforms,
protein-level effects and competition between multiplexed guides are not modeled.

## Unresolved equilibria and numerical verification

In [8]:
coverage = summary[summary.condition.isin(CORE)][['network','condition','n_pairs',
    'n_single_converged','n_double_converged','n_valid_pairs','pair_any_gt_0.1']].copy()
coverage['unresolved_pair_fraction'] = 1 - coverage.n_valid_pairs/coverage.n_pairs
display(coverage.round(4))
for filename in ['numerical_validation.csv','adaptive_ode_validation.csv']:
    path = RESULTS / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path).round(8))
    else:
        print('Run: python job_script/validate_modalities.py')

numerical_validation.csv
adaptive_ode_validation.csv


,network,condition,n_pairs,n_single_converged,n_double_converged,n_valid_pairs,pair_any_gt_0.1,unresolved_pair_fraction
0,V1,CRISPRa_10x,4096,256,4096,4096,0.0862,0.0000
2,V1,CRISPRa_2x,4096,256,4096,4096,0.0142,0.0000
5,V1,KD_50pct,4096,256,4096,4096,0.0046,0.0000
6,V1,KD_90pct,4096,256,4096,4096,0.0154,0.0000
8,V1,KO,4096,256,4096,4096,0.0200,0.0000
9,V1,OE_10x,4096,256,4096,4096,0.0574,0.0000
11,V1,OE_30x,4096,256,4096,4096,0.0833,0.0000
19,V3,CRISPRa_10x,4096,207,2999,2515,0.3917,0.3860
21,V3,CRISPRa_2x,4096,214,2980,2785,0.1903,0.3201
24,V3,KD_50pct,4096,222,3186,3052,0.1507,0.2549


,network,condition,kind,checked,comparable,convergence_agreement,recovered_with_longer_run,max_expression_difference,max_log2_difference_above_floor
0,V1,KO,double,10,10,10,0,1.000000e-08,4.000000e-08
1,V1,KO,single,19,19,19,0,1.000000e-08,4.000000e-08
2,V1,KD_90pct,double,10,10,10,0,1.000000e-08,3.000000e-08
3,V1,KD_90pct,single,19,19,19,0,1.000000e-08,4.000000e-08
4,V1,CRISPRa_10x,double,10,10,10,0,2.100000e-07,9.100000e-07
5,V1,CRISPRa_10x,single,19,19,19,0,3.000000e-08,9.000000e-08
6,V1,OE_30x,double,10,10,10,0,3.000000e-08,2.000000e-08
7,V1,OE_30x,single,18,18,18,0,3.000000e-08,1.200000e-07
8,V3,KO,double,12,8,12,0,1.000000e-08,6.000000e-08
9,V3,KO,single,22,18,22,0,1.000000e-08,6.000000e-08


,network,condition,pair_index,success,final_max_drift,max_log2_difference_above_floor
0,V1,KO,217,True,0.0,1.000000e-08
1,V1,KO,213,True,0.0,3.000000e-08
2,V1,KD_90pct,213,True,0.0,1.000000e-08
3,V1,KD_90pct,217,True,0.0,3.000000e-08
4,V1,CRISPRa_10x,41,True,0.0,1.000000e-08
5,V1,CRISPRa_10x,712,True,0.0,1.010000e-06
6,V1,OE_30x,245,True,0.0,2.000000e-08
7,V1,OE_30x,0,True,0.0,0.000000e+00
8,V3,KO,1140,True,0.0,3.000000e-08
9,V3,KO,643,True,0.0,5.000000e-08


No-convergence is an outcome, not an additive result. The default horizon is 1,000 model-time
units; the original notebook's maximum is 200. Refined checks halve the integration step,
double that horizon, and tighten tolerances tenfold. Independent DOP853 checks use the same
ODE from the same WT state. A longer horizon can recover slow transients. Disagreement on a
particular attractor is reported rather than hidden; this toy model can have multiple basins.

The original `ko_nodes` convergence flag was the saved WT flag because perturbations used
`save=False`. That bug is fixed. The older time-average convergence criterion itself remains
available in `simulate_rna`; the new equilibrium solver checks the drift of every free gene.

## Null calibration with the original stochastic simulator

Three independent groups of four unperturbed trajectories per network were treated as WT, A, B and AB, using the original `s=1e-4`, time step and observation model. This measures a small null noise sample. It does not test noise-induced basin switching under every perturbation.

In [9]:
display(pd.read_csv(RESULTS / 'noise_null.csv').round(7))

,network,replicate,eligible_genes,p99_abs_residual,max_abs_residual,fraction_gt_0p1
0,V1,0,256,0.000892,0.002685,0.0
1,V1,1,256,0.001368,0.006577,0.0
2,V1,2,256,0.001093,0.002349,0.0
3,V3,0,210,0.002152,0.003362,0.0
4,V3,1,210,0.002397,0.005561,0.0
5,V3,2,210,0.004589,0.007153,0.0


## Additional network seeds

In [10]:
replicate_paths = list((ROOT / 'results' / 'seed_sensitivity').glob('summary*.csv'))
if replicate_paths:
    replicates = pd.concat([pd.read_csv(p) for p in replicate_paths], ignore_index=True)
    display(replicates[['network','seed','condition','n_pairs','n_valid_pairs',
                        'readout_gt_0.1','pair_any_gt_0.1']].round(4))
else:
    print('Additional seed checks not present.')

,network,seed,condition,n_pairs,n_valid_pairs,readout_gt_0.1,pair_any_gt_0.1
0,V1,5,KO,1024,1024,0.0038,0.0371
1,V1,5,KD_90pct,1024,1024,0.0034,0.0332
2,V1,5,CRISPRa_10x,1024,1024,0.0145,0.0967
3,V1,5,OE_30x,1024,1024,0.0174,0.0928
4,V1,7,KO,1024,1024,0.0012,0.0215
5,V1,7,KD_90pct,1024,1024,0.0008,0.0215
6,V1,7,CRISPRa_10x,1024,1024,0.0048,0.0684
7,V1,7,OE_30x,1024,1024,0.0066,0.0586
8,V3,3,KO,512,342,0.0887,0.2719
9,V3,3,KD_90pct,512,351,0.0911,0.2764


These checks address dependence on the particular toy realization. They do not establish
the prevalence in real GRNs. V1's label “buffered” and V3's label “propagating” describe the
chosen model regimes; “realistic” requires a separate calibration to experimental data.

For prediction work, use matched train/test pairs across modalities, exclude perturbed
genes from evaluation, and compare both expression-additive and log-additive baselines.
Report performance on all readouts and the subset affected by both singles. Large interaction
tails create headroom for a model but do not establish that an MLP can learn those tails.